In [8]:
import os
from pathlib import Path
import time
import json
import shutil
import pickle
import traceback
import argparse
import warnings
import boto3
import pprint


In [3]:
pp = pprint.PrettyPrinter(width=41, compact=True, indent=4)
warnings.filterwarnings('ignore')

logs = []

In [12]:
def prepare_directories(root):
    try:
        # input
        input_dir = os.path.join(root, 'input')
        os.makedirs(os.path.join(input_dir, 'profile'), exist_ok=True)
        os.makedirs(os.path.join(input_dir, 'data'), exist_ok=True) # input_data_ref.yml
        os.makedirs(os.path.join(input_dir, 'metadata'), exist_ok=True) # run_manifest.yml
        os.makedirs(os.path.join(input_dir, 'config'), exist_ok=True) # config.yml
        return input_dir
    except Exception as e:
        pp.pprint(e)
        logs.append(str(e))
        pass

In [13]:
root = str(Path().resolve())

In [14]:
prepare_directories(root)

'/home/ec2-user/SageMaker/gs-ds-env/tests/titanic/notebooks/sean/run_pm/input'

# s3 예제 위치
```
s3://gs-retail-awesome-model/env=dev/project=gs25-sales-forecast/experiment=baseline-v1/model=store-daily-sales-forecast/algo=lightgbm/run_date=2026-02-28/run_id=20260228T143022Z_a7f3c8d1/
```

In [17]:
"""
prepare_datasets.ipynb - SageMaker Training Input 데이터 준비 및 S3 업로드
"""

# %% [markdown]
# # SageMaker Training Input 데이터 준비
# 
# 이 노트북은 training job에 필요한 input 파일들을 생성하고 S3에 업로드합니다.

# %% Imports
import os
import boto3
import yaml
from datetime import datetime
from pathlib import Path
import tempfile
import shutil

# %% Configuration
class TrainingConfig:
    """Training job 설정"""
    
    # S3 설정
    BUCKET = "gs-retail-awesome-model"
    ENV = "dev"
    PROJECT = "gs25-sales-forecast"
    EXPERIMENT = "baseline-v1"
    MODEL = "store-daily-sales-forecast"
    ALGO = "lightgbm"
    
    @classmethod
    def generate_run_id(cls):
        """고유한 run_id 생성"""
        import uuid
        now = datetime.utcnow()
        short_uuid = uuid.uuid4().hex[:8]
        return now.strftime(f"%Y%m%dT%H%M%SZ_{short_uuid}")
    
    @classmethod
    def get_s3_prefix(cls, run_date: str = None, run_id: str = None):
        """S3 prefix 경로 생성"""
        if run_date is None:
            run_date = datetime.utcnow().strftime("%Y-%m-%d")
        if run_id is None:
            run_id = cls.generate_run_id()
        
        return (
            f"env={cls.ENV}/"
            f"project={cls.PROJECT}/"
            f"experiment={cls.EXPERIMENT}/"
            f"model={cls.MODEL}/"
            f"algo={cls.ALGO}/"
            f"run_date={run_date}/"
            f"run_id={run_id}"
        )

# %% Directory & File Preparation Functions
def prepare_local_directories(root: str) -> dict:
    """
    로컬에 input 디렉토리 구조 생성
    
    Returns:
        dict: 생성된 디렉토리 경로들
    """
    dirs = {
        'root': root,
        'input': os.path.join(root, 'input'),
        'profile': os.path.join(root, 'input', 'profile'),
        'data': os.path.join(root, 'input', 'data'),
        'metadata': os.path.join(root, 'input', 'metadata'),
        'config': os.path.join(root, 'input', 'config'),
    }
    
    for name, path in dirs.items():
        os.makedirs(path, exist_ok=True)
        print(f"✓ Created: {path}")
    
    return dirs


def create_input_data_ref(data_dir: str, config: dict = None) -> str:
    """
    input_data_ref.yml 생성 - 학습 데이터 참조 정보
    """
    default_config = {
        'version': '1.0',
        'created_at': datetime.utcnow().isoformat(),
        'data_sources': {
            'train': {
                's3_uri': 's3://gs-retail-data-lake/processed/sales/train/',
                'format': 'parquet',
                'partition_keys': ['store_id', 'date'],
            },
            'validation': {
                's3_uri': 's3://gs-retail-data-lake/processed/sales/validation/',
                'format': 'parquet',
                'partition_keys': ['store_id', 'date'],
            },
            'features': {
                's3_uri': 's3://gs-retail-data-lake/features/store-features/',
                'format': 'parquet',
            }
        },
        'schema': {
            'target_column': 'daily_sales',
            'feature_columns': [
                'store_id', 'day_of_week', 'is_holiday', 
                'temperature', 'precipitation', 'promotion_flag'
            ],
            'id_columns': ['store_id', 'date']
        }
    }
    
    final_config = config if config else default_config
    filepath = os.path.join(data_dir, 'input_data_ref.yml')
    
    with open(filepath, 'w', encoding='utf-8') as f:
        yaml.dump(final_config, f, default_flow_style=False, allow_unicode=True)
    
    print(f"✓ Created: {filepath}")
    return filepath


def create_run_manifest(metadata_dir: str, config: dict = None) -> str:
    """
    run_manifest.yml 생성 - 실행 메타데이터
    """
    default_config = {
        'version': '1.0',
        'run_info': {
            'created_at': datetime.utcnow().isoformat(),
            'created_by': 'ax-squad',
            'environment': TrainingConfig.ENV,
            'project': TrainingConfig.PROJECT,
            'experiment': TrainingConfig.EXPERIMENT,
        },
        'execution': {
            'instance_type': 'ml.m5.xlarge',
            'instance_count': 1,
            'max_runtime_seconds': 3600,
            'volume_size_gb': 30,
        },
        'tracking': {
            'mlflow_experiment_name': f"{TrainingConfig.PROJECT}/{TrainingConfig.EXPERIMENT}",
            'tags': {
                'team': 'ax-squad',
                'cost_center': 'ai-transformation',
            }
        }
    }
    
    final_config = config if config else default_config
    filepath = os.path.join(metadata_dir, 'run_manifest.yml')
    
    with open(filepath, 'w', encoding='utf-8') as f:
        yaml.dump(final_config, f, default_flow_style=False, allow_unicode=True)
    
    print(f"✓ Created: {filepath}")
    return filepath


def create_config(config_dir: str, config: dict = None) -> str:
    """
    config.yml 생성 - 모델/알고리즘 하이퍼파라미터
    """
    default_config = {
        'version': '1.0',
        'algorithm': {
            'name': 'lightgbm',
            'task': 'regression',
        },
        'hyperparameters': {
            'objective': 'regression',
            'metric': 'rmse',
            'boosting_type': 'gbdt',
            'num_leaves': 31,
            'learning_rate': 0.05,
            'feature_fraction': 0.9,
            'bagging_fraction': 0.8,
            'bagging_freq': 5,
            'n_estimators': 1000,
            'early_stopping_rounds': 50,
            'verbose': -1,
        },
        'preprocessing': {
            'handle_missing': 'fill_median',
            'categorical_encoding': 'label',
            'scale_features': False,
        },
        'cross_validation': {
            'strategy': 'time_series_split',
            'n_splits': 5,
        }
    }
    
    final_config = config if config else default_config
    filepath = os.path.join(config_dir, 'config.yml')
    
    with open(filepath, 'w', encoding='utf-8') as f:
        yaml.dump(final_config, f, default_flow_style=False, allow_unicode=True)
    
    print(f"✓ Created: {filepath}")
    return filepath


def create_profile(profile_dir: str, config: dict = None) -> str:
    """
    profile.yml 생성 - 실행 환경 프로파일
    """
    default_config = {
        'version': '1.0',
        'profile': {
            'name': 'default',
            'description': 'Default training profile for GS25 sales forecast',
        },
        'aws': {
            'region': 'ap-northeast-2',
            'role_arn': 'arn:aws:iam::123456789012:role/SageMakerExecutionRole',
        },
        'python': {
            'version': '3.10',
            'dependencies': [
                'lightgbm==4.1.0',
                'pandas==2.0.3',
                'numpy==1.24.3',
                'scikit-learn==1.3.0',
                'pyyaml==6.0.1',
            ]
        }
    }
    
    final_config = config if config else default_config
    filepath = os.path.join(profile_dir, 'profile.yml')
    
    with open(filepath, 'w', encoding='utf-8') as f:
        yaml.dump(final_config, f, default_flow_style=False, allow_unicode=True)
    
    print(f"✓ Created: {filepath}")
    return filepath


# %% S3 Upload Functions
def upload_directory_to_s3(local_dir: str, bucket: str, s3_prefix: str, dry_run: bool = False) -> list:
    """
    로컬 디렉토리를 S3에 업로드
    
    Args:
        local_dir: 업로드할 로컬 디렉토리
        bucket: S3 버킷명
        s3_prefix: S3 prefix
        dry_run: True면 실제 업로드 없이 경로만 출력
    
    Returns:
        list: 업로드된 S3 경로들
    """
    s3_client = boto3.client('s3')
    uploaded_files = []
    
    for root, dirs, files in os.walk(local_dir):
        for filename in files:
            local_path = os.path.join(root, filename)
            relative_path = os.path.relpath(local_path, local_dir)
            s3_key = f"{s3_prefix}/{relative_path}"
            
            s3_uri = f"s3://{bucket}/{s3_key}"
            
            if dry_run:
                print(f"[DRY RUN] Would upload: {local_path} -> {s3_uri}")
            else:
                s3_client.upload_file(local_path, bucket, s3_key)
                print(f"✓ Uploaded: {s3_uri}")
            
            uploaded_files.append(s3_uri)
    
    return uploaded_files


# %% Main Execution
def prepare_and_upload(
    run_date: str = None,
    run_id: str = None,
    dry_run: bool = False,
    custom_configs: dict = None
) -> dict:
    """
    전체 프로세스 실행: 파일 생성 및 S3 업로드
    
    Args:
        run_date: 실행 날짜 (YYYY-MM-DD), None이면 오늘
        run_id: 실행 ID, None이면 자동 생성
        dry_run: True면 실제 업로드 없이 시뮬레이션
        custom_configs: 각 파일별 커스텀 설정 dict
    
    Returns:
        dict: 결과 정보
    """
    custom_configs = custom_configs or {}
    
    # 1. Run 정보 생성
    if run_date is None:
        run_date = datetime.utcnow().strftime("%Y-%m-%d")
    if run_id is None:
        run_id = TrainingConfig.generate_run_id()
    
    s3_prefix = TrainingConfig.get_s3_prefix(run_date, run_id)
    
    print("=" * 60)
    print("SageMaker Training Input 데이터 준비")
    print("=" * 60)
    print(f"Run Date: {run_date}")
    print(f"Run ID: {run_id}")
    print(f"S3 URI: s3://{TrainingConfig.BUCKET}/{s3_prefix}/")
    print("=" * 60)
    
    # 2. 임시 디렉토리에 파일 생성
    with tempfile.TemporaryDirectory() as tmp_dir:
        print("\n[Step 1] 로컬 디렉토리 구조 생성")
        dirs = prepare_local_directories(tmp_dir)
        
        print("\n[Step 2] 설정 파일 생성")
        create_input_data_ref(dirs['data'], custom_configs.get('data_ref'))
        create_run_manifest(dirs['metadata'], custom_configs.get('manifest'))
        create_config(dirs['config'], custom_configs.get('config'))
        create_profile(dirs['profile'], custom_configs.get('profile'))
        
        print("\n[Step 3] S3 업로드")
        uploaded = upload_directory_to_s3(
            local_dir=dirs['input'],
            bucket=TrainingConfig.BUCKET,
            s3_prefix=f"{s3_prefix}/input",
            dry_run=dry_run
        )
    
    result = {
        'run_date': run_date,
        'run_id': run_id,
        's3_prefix': s3_prefix,
        's3_uri': f"s3://{TrainingConfig.BUCKET}/{s3_prefix}/",
        'uploaded_files': uploaded,
        'dry_run': dry_run
    }
    
    print("\n" + "=" * 60)
    print("완료!")
    print(f"S3 Input Path: {result['s3_uri']}input/")
    print("=" * 60)
    
    return result

In [18]:
result = prepare_and_upload(dry_run=True)

SageMaker Training Input 데이터 준비
Run Date: 2026-03-02
Run ID: 20260302T142925Z_900c70f6
S3 URI: s3://gs-retail-awesome-model/env=dev/project=gs25-sales-forecast/experiment=baseline-v1/model=store-daily-sales-forecast/algo=lightgbm/run_date=2026-03-02/run_id=20260302T142925Z_900c70f6/

[Step 1] 로컬 디렉토리 구조 생성
✓ Created: /tmp/tmpeuzhf7wn
✓ Created: /tmp/tmpeuzhf7wn/input
✓ Created: /tmp/tmpeuzhf7wn/input/profile
✓ Created: /tmp/tmpeuzhf7wn/input/data
✓ Created: /tmp/tmpeuzhf7wn/input/metadata
✓ Created: /tmp/tmpeuzhf7wn/input/config

[Step 2] 설정 파일 생성
✓ Created: /tmp/tmpeuzhf7wn/input/data/input_data_ref.yml
✓ Created: /tmp/tmpeuzhf7wn/input/metadata/run_manifest.yml
✓ Created: /tmp/tmpeuzhf7wn/input/config/config.yml
✓ Created: /tmp/tmpeuzhf7wn/input/profile/profile.yml

[Step 3] S3 업로드
[DRY RUN] Would upload: /tmp/tmpeuzhf7wn/input/config/config.yml -> s3://gs-retail-awesome-model/env=dev/project=gs25-sales-forecast/experiment=baseline-v1/model=store-daily-sales-forecast/algo=lightgbm/ru

In [20]:
# 실제 업로드
result = prepare_and_upload(dry_run=False)

# 커스텀 설정으로 실행
custom = {
    'config': {
        'hyperparameters': {'learning_rate': 0.1, 'n_estimators': 500}
    }
}
result = prepare_and_upload(custom_configs=custom, dry_run=False)

SageMaker Training Input 데이터 준비
Run Date: 2026-03-02
Run ID: 20260302T142959Z_44d33750
S3 URI: s3://gs-retail-awesome-model/env=dev/project=gs25-sales-forecast/experiment=baseline-v1/model=store-daily-sales-forecast/algo=lightgbm/run_date=2026-03-02/run_id=20260302T142959Z_44d33750/

[Step 1] 로컬 디렉토리 구조 생성
✓ Created: /tmp/tmpnl7_t1mh
✓ Created: /tmp/tmpnl7_t1mh/input
✓ Created: /tmp/tmpnl7_t1mh/input/profile
✓ Created: /tmp/tmpnl7_t1mh/input/data
✓ Created: /tmp/tmpnl7_t1mh/input/metadata
✓ Created: /tmp/tmpnl7_t1mh/input/config

[Step 2] 설정 파일 생성
✓ Created: /tmp/tmpnl7_t1mh/input/data/input_data_ref.yml
✓ Created: /tmp/tmpnl7_t1mh/input/metadata/run_manifest.yml
✓ Created: /tmp/tmpnl7_t1mh/input/config/config.yml
✓ Created: /tmp/tmpnl7_t1mh/input/profile/profile.yml

[Step 3] S3 업로드


S3UploadFailedError: Failed to upload /tmp/tmpnl7_t1mh/input/config/config.yml to gs-retail-awesome-model/env=dev/project=gs25-sales-forecast/experiment=baseline-v1/model=store-daily-sales-forecast/algo=lightgbm/run_date=2026-03-02/run_id=20260302T142959Z_44d33750/input/config/config.yml: An error occurred (NoSuchBucket) when calling the PutObject operation: The specified bucket does not exist